# McKenzie et al. (2025) on dandiset 000552: reactivation in ripples (Figure 8), the familiar run (fig8d, and fig8-no-pre in the familiar run)

Paper: McKenzie et al. (2025), eLife 14:RP105183, [doi:10.7554/elife.105183](https://doi.org/10.7554/elife.105183);
claims file `claims/mckenzie-2025.yaml`, written from Reviewed Preprint version 1. The novel and familiar
runs are in two notebooks, `reactivation.ipynb` and `reactivation_familiar.ipynb`, to keep each under an
hour; the sessions, the ripple detection and the procedure are the same in both.

| Claim | Here |
|---|---|
| fig8d-familiar-no-difference | tested |
| fig8-no-pre-reactivation | tested in the familiar run; the novel run is in `reactivation.ipynb` |
| fig8b-late-moments-reactivated | in `reactivation.ipynb` |

Dataset: [DANDI:000552](https://dandiarchive.org/dandiset/000552/0.230630.2304) version **0.230630.2304**, the
version the paper cites (Methods, Electrophysiology subjects). Pyramidal cells are taken from the
CellExplorer labels of the later version 0.261113.1035, after checking that its spike times are the
same; the pinned version has no cell types. The sessions and their checks are in `settling.ipynb`.

Method (Methods, Ripple detection and Reactivation analysis; claims file E9, E10). Ripples are detected as
the Methods say from the 1250 Hz LFP in each session's raw file (`processing/ecephys/LFP`, stored one
channel per chunk, so one channel can be streamed): 130 to 200 Hz fourth-order Chebyshev, normalized
squared signal, peaks above 5 SD, edges at 2 SD, 20 to 200 ms. The channel is the one with the largest
ripple-band power over 165 s of home cage before the familiar run, taken as the pyramidal layer (a
standard choice; the Methods name none). The ripples Huszar et al. deposited in the files are a variant
(E9). Each ripple's vector is each pyramidal unit's spike count over the ripple divided by its duration;
it is correlated with the 100 ms vectors of the run; the null moves each unit's value on a ripple to a
random other ripple (an independent permutation of each unit across the pool), 1000 times; a ripple
reactivates a moment when its R is above all 1000 shuffles (above their 99.9th percentile). The moments
are the bins of minute 1 (0 to 60 s) and of minutes 10 to 11 (600 to 660 s, Figure 8B), 1200 per run.

Readings of E10, fully crossed: pre ripples (a) the 30 min before the run, (b) the first 30 min of the
recording; post ripples the 30 min after the run; track vectors z-scored over the first 1000 s or raw
counts; shuffle pool (a) the set the ripple belongs to, (b) pre and post together. The main reading is pre
(a), z-scored, pool (a). Under pool (a) the post ripples do not depend on the pre window, so there are six
readings for post ripples and eight for pre ripples.

In [ ]:
import time, zlib, warnings, resource
import numpy as np
import pandas as pd
from scipy import stats
import settle as S

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200, "display.max_columns", 40)
T_START = time.time()
ENV = "fam"

def seed(claim, k=0):
    return zlib.crc32(f"mckenzie-2025/{claim}".encode()) + k

SESS = S.reactivation_sessions()
TEST = [sid for sid in SESS if sid != S.B_SID]
print(f"{len(TEST)} sessions loaded and ripples detected in {time.time() - T_START:.0f} s:", TEST)

In [ ]:
# the detected ripples against those Huszar et al. deposited: the share of detected peaks inside a deposited
# ripple, and the same with the peaks shifted by 1 s (a check that the LFP and the spikes share a clock)
def inside(peaks, rip):
    i = np.searchsorted(rip[:, 0], peaks) - 1
    return float(np.mean((i >= 0) & (peaks <= rip[np.clip(i, 0, None), 1])))

rows = []
for sid in TEST:
    s = SESS[sid]; ri = s["rip_info"]; dur = s["rip_det"][:, 1] - s["rip_det"][:, 0]
    rows.append(dict(session=sid, lfp_file=ri["source"], channel=ri["channel"], of=ri["channels"], power_vs_median_channel=round(ri["power_vs_median"], 2),
                     detected=len(s["rip_det"]), deposited=len(s["rip"]),
                     detected_in_deposited=round(inside(ri["peaks"], s["rip"]), 3),
                     shifted_1s=round(inside(ri["peaks"] + 1.0, s["rip"]), 3),
                     dur_ms=f"{dur.min()*1e3:.0f} to {dur.max()*1e3:.0f} (median {np.median(dur)*1e3:.0f})"))
print(pd.DataFrame(rows).to_string())

WIN = {}
rows = []
for sid, s in SESS.items():
    r = s["runs"][1 if ENV == "nov" else 0]
    w = S.reactivation_windows(s, r)
    WIN[sid] = w
    row = dict(session=sid, env=ENV, t0=round(r["t0"]), t1=round(r["t1"]))
    for k, (a, b) in w.items():
        row[f"detected_{k}"] = int(((s["rip_det"][:, 0] >= a) & (s["rip_det"][:, 0] < b)).sum())
        row[f"deposited_{k}"] = int(((s["rip"][:, 0] >= a) & (s["rip"][:, 0] < b)).sum())
        row[f"{k}_window"] = f"{a:.0f}-{b:.0f}"
    # the post window must not run into the next maze run
    nxt = [x["t0"] for x in s["runs"] if x["t0"] > r["t0"]]
    row["post_overlaps_next_run_s"] = round(max(0.0, r["t1"] + 1800 - nxt[0]), 1) if nxt else 0.0
    row["pyramidal"] = int((s["cell_type"] == "Pyramidal Cell").sum())
    rows.append(row)
print(pd.DataFrame(rows).to_string())

In [ ]:
MOM, MOM_FT = {}, {}
for sid, s in SESS.items():
    pyr = np.where(s["cell_type"] == "Pyramidal Cell")[0]
    r = s["runs"][1 if ENV == "nov" else 0]
    MOM[sid] = S.run_moments(s, r, pyr)
    MOM_FT[sid] = S.run_moments(s, r, pyr, off=r["first_track"] - r["t0"])   # variant: from the first tracked frame

def run_reactivation(sid, rng, pre="preA", vec="z", pool="a", sets=("pre", "post"), mom=MOM, rip="rip_det"):
    s = SESS[sid]
    return [dict(session=sid, env=ENV, **r) for r in S.run_reactivation(s, s[rip], WIN[sid], mom[sid], rng, pre, vec, pool, sets)]

# the readings of E10, crossed; under pool (a) the post ripples do not depend on the pre window
READINGS = [("preA", "z", "a", ("pre", "post")), ("preB", "z", "a", ("pre",)),
            ("preA", "raw", "a", ("pre", "post")), ("preB", "raw", "a", ("pre",)),
            ("preA", "z", "b", ("pre", "post")), ("preB", "z", "b", ("pre", "post")),
            ("preA", "raw", "b", ("pre", "post")), ("preB", "raw", "b", ("pre", "post"))]

def readings(claim, sessions):
    rng = np.random.default_rng(seed(claim))
    return pd.DataFrame([r for sid in sessions for pre, vec, pool, sets in READINGS for r in run_reactivation(sid, rng, pre, vec, pool, sets)])

def late_vs_early(df, sessions):
    out = []
    for key, d in df[(df.set == "post") & df.session.isin(sessions)].groupby(["pre", "vec", "pool"]):
        if key[0] == "preB" and key[2] == "a":
            continue
        t, p = stats.ttest_rel(d.late_pct, d.early_pct)
        out.append(dict(zip(["pre", "vec", "pool"], key), early_mean=d.early_pct.mean(), late_mean=d.late_pct.mean(),
                        diff=(d.late_pct - d.early_pct).mean(), t=t, df=len(d) - 1, p=p))
    return pd.DataFrame(out)

def late_early(d, label):
    t, p = stats.ttest_rel(d.late_pct, d.early_pct)
    print(f"{label}: early {d.early_pct.mean():.4f}%, late {d.late_pct.mean():.4f}%, late - early = {(d.late_pct - d.early_pct).mean():+.4f}, t({len(d) - 1}) = {t:.2f}, p = {p:.3g}")

def pre_test(d, label):
    t, p = stats.ttest_1samp(d.all_pct, 0.1, alternative="greater")
    print(f"{label}: mean {d.all_pct.mean():.4f}% (sessions {d.all_pct.min():.4f} to {d.all_pct.max():.4f}), one-sided t({len(d) - 1}) = {t:.2f}, p = {p:.3g}")

## fig8d-familiar-no-difference

> In the familiar environment, early and late moments do not differ in reactivation rate.

Criterion: a claim of absence, judged on the paired t-test over sessions at p < 0.05, on the familiar
figure-eight run with pre and post ripples around it, under E10 (a) with z-scored vectors; the criterion
names no shuffle pool, and pool (a), the first the claims file lists, is used. The other readings are shown.

In [ ]:
t0 = time.time()
SETS = {"8 sessions": TEST}
R = readings("fig8d-familiar-no-difference", TEST)
print(f"readings in {time.time() - t0:.0f} s")
print(R[R.set == "post"].round(3).to_string())
for name, sset in SETS.items():
    print(name)
    print(late_vs_early(R, sset).round(4).to_string())

In [ ]:
t0 = time.time()
main = R[(R.set == "post") & (R.pre == "preA") & (R.vec == "z") & (R.pool == "a") & R.session.isin(TEST)]
late_early(main, "main reading, claim seed")
for k in range(1, 5):
    rng = np.random.default_rng(seed("fig8d-familiar-no-difference") + 1000 * k)
    d = pd.DataFrame([r for sid in TEST for r in run_reactivation(sid, rng, sets=("post",))])
    late_early(d, f"main reading, seed + {1000 * k}")
print(f"seeds in {time.time() - t0:.0f} s")
# variants, not judged: the deposited ripples (E9), and time 0 at the first tracked frame (E2)
rng = np.random.default_rng(seed("fig8d-familiar-no-difference") + 7)
DEP = pd.DataFrame([r for sid in TEST for r in run_reactivation(sid, rng, rip="rip")])
late_early(DEP[DEP.set == "post"], "deposited ripples, main reading")
rng = np.random.default_rng(seed("fig8d-familiar-no-difference") + 8)
FT = pd.DataFrame([r for sid in TEST for r in run_reactivation(sid, rng, sets=("post",), mom=MOM_FT)])
late_early(FT, "from the first tracked frame, main reading")

**Result: not reproduced.** The judged test finds a difference in the familiar run, with early moments
reactivated more than late ones: 0.251% against 0.224%, -0.0274 percentage points, t(7) = -3.29, p = 0.0134,
and p 0.0099 to 0.0338 under the five seeds; with the deposited ripples -0.0251 (p = 0.0065). The other
readings find no difference: raw counts p = 0.719, and 0.981 and 0.856 with pooled shuffles; z-scored with
pooled shuffles p = 0.562 and 0.461. With time 0 at the first tracked frame, a variant of claims file E2 not
judged, there is no difference (+0.0223, t(7) = 1.15, p = 0.286). The difference is about a tenth of the
reactivation rate, and opposite in sign to the novel effect the paper reports.

## fig8-no-pre-reactivation, familiar run

> Track activity patterns show no evidence of being present in ripples recorded before the experience.

Criterion: per session the mean over moments of the percentage of pre ripples that reactivate each moment;
reproduced when its mean over sessions is not above 0.1% by a one-sided one-sample t-test at p < 0.05, in
novel and familiar runs (the novel run is in `reactivation.ipynb`). Main reading: pre (a), z-scored, pool (a), under
this claim's own seed and four more; the other readings are those run above.

In [ ]:
t0 = time.time()
k_env = 1       # the novel and familiar runs take different streams of the claim's seeds
for k in range(5):
    rng = np.random.default_rng(seed("fig8-no-pre-reactivation") + 1000 * k + k_env)
    d = pd.DataFrame([r for sid in TEST for r in run_reactivation(sid, rng, sets=("pre",))])
    if k == 0:
        PRE = d
        print(d.round(4).to_string())
    pre_test(d, f"main reading, seed + {1000 * k + k_env}")
print(f"pre ripples in {time.time() - t0:.0f} s")
for name, sset in SETS.items():
    for key, d in R[(R.set == "pre") & R.session.isin(sset)].groupby(["pre", "vec", "pool"]):
        pre_test(d, f"{name}, {key}, claim seed of fig8d-familiar-no-difference")
pre_test(DEP[DEP.set == "pre"], "deposited ripples, main reading")
post = R[(R.set == "post") & (R.pre == "preA") & (R.vec == "z") & (R.pool == "a") & R.session.isin(TEST)]
print(f"post ripples for comparison: mean {post.all_pct.mean():.4f}%")

In [ ]:
t0 = time.time()
# the two windows of the main reading separately, and two variants not judged: time 0 at the first tracked
# frame (claims file E2), and the mean over every moment of the run, not only the 1200 of Figure 8B
for col, label in (("early_pct", "minute 1 only"), ("late_pct", "minutes 10 to 11 only")):
    pre_test(PRE.assign(all_pct=PRE[col]), f"main reading, {label}")
rng = np.random.default_rng(seed("fig8-no-pre-reactivation") + 7 + k_env)
FTP = pd.DataFrame([r for sid in TEST for r in run_reactivation(sid, rng, sets=("pre",), mom=MOM_FT)])
print(FTP.round(4).to_string())
pre_test(FTP, "from the first tracked frame, main reading")
pre_test(FTP.assign(all_pct=FTP.late_pct), "from the first tracked frame, minutes 10 to 11 only")
MOM_ALL = {}
for sid in TEST:
    s = SESS[sid]
    MOM_ALL[sid] = S.run_moments(s, s["runs"][1 if ENV == "nov" else 0], np.where(s["cell_type"] == "Pyramidal Cell")[0], whole=True)
rng = np.random.default_rng(seed("fig8-no-pre-reactivation") + 9 + k_env)
WH = pd.DataFrame([r for sid in TEST for r in run_reactivation(sid, rng, sets=("pre",), mom=MOM_ALL)])
WH["moments"] = [len(MOM_ALL[sid]["z"]) for sid in WH.session]
print(WH.round(4).to_string())
pre_test(WH, "whole run from the epoch start, main reading")
pre_test(WH.assign(all_pct=WH.late_pct), "whole run after minute 1, main reading")
print(f"variants in {time.time() - t0:.0f} s")


**Result, familiar run: pre ripples reactivate track moments above the false positive rate.** Under the main
reading pre ripples reactivate on average 0.1491% of moments (sessions 0.1216 to 0.1670), above 0.1% (one-sided
t(7) = 9.71, p = 1.3e-05), with p below 0.0001 under all five seeds, p of 0.0017 or less under every other
reading, and p = 6.5e-05 with the deposited ripples. Post ripples reactivate more (0.2375%). Minute 1 is
the highest here too (0.1856%, p = 9.7e-06), but minutes 10 to 11 alone are also above 0.1% (0.1125%, p =
0.0116). Two variants, not judged: with time 0 at the first tracked frame (claims file E2) 0.1273% (p =
8.1e-05); over every moment of the run (12630 to 23743 per session) 0.1187% (sessions 0.0957 to 0.1374,
one-sided t(7) = 4.17, p = 0.0021), and 0.1166% after minute 1 (p = 0.0035). In the familiar run the pre
ripples are above 0.1% under every variant. With the novel run
(`reactivation.ipynb`, p = 0.0086) the claim is not reproduced: the track patterns appear in post ripples more
than in pre ripples, but they are not absent from pre ripples.

## Summary

| Claim | Outcome |
|---|---|
| fig8d-familiar-no-difference | not reproduced |
| fig8-no-pre-reactivation | not reproduced (above the false positive rate here and in `reactivation.ipynb`) |
| fig8b-late-moments-reactivated | in `reactivation.ipynb` |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

Differences from the paper's method are those of `reactivation.ipynb`. The fig8d outcome rests on the shuffle
pool, which the criterion does not name: with pre and post pooled the difference is not significant.

In [ ]:
print(f"notebook runtime {(time.time() - T_START) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.1f} GB")